In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [1]:
import pandas as pd
import numpy as np
import csv
import pickle
import copy
import re
import random
import matplotlib.pyplot as plt
import itertools
import json
import openai
import time
from bs4 import BeautifulSoup


!pip install ipython-autotime
%load_ext autotime



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.6 MB/s eta 0:00:00
time: 283 µs (started: 2026-04-24 02:30:49 +00:00)


In [2]:

!rm -rf LLM4BEAR
!git clone --depth 1 --filter=blob:none --sparse https://github.com/anon5159753/LLM4BEAR.git
!cd LLM4BEAR && git sparse-checkout add "3_Human Evaluation"

Cloning into 'LLM4BEAR'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (64/64), done.
Receiving objects: 100% (67/67), 39.56 KiB | 5.65 MiB/s, done.
remote: Total 67 (delta 6), reused 39 (delta 0), pack-reused 0 (from 0)
Resolving deltas: 100% (6/6), done.
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 5.44 KiB | 2.72 MiB/s, done.
remote: Enumerating objects: 448, done.
remote: Counting objects: 100% (448/448), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 448 (delta 364), reused 443 (delta 364), pack-reused 0 (from 0)
Receiving objects: 100% (448/448), 4.72 MiB | 18.25 MiB/s, done.
Resolving deltas: 100% (364/364), done.
Updating files: 100% (452/452), done.
time: 2.85 s (started: 2026-04-24 02:30:49 +00:00)


#Forms completed by independent participants

In [ ]:
clothing_forms = [1, 3, 4, 5, 6, 7, 10, 11, 13, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 29, 30, 31, 32, 33, 34, 35, 37, 38]
electronic_forms = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 20, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 34, 35, 36, 39]
food_forms = [1, 2, 4, 5, 6, 7, 9, 10, 11, 12, 14, 15, 19, 20, 21, 23, 24, 27, 28, 31, 33, 34, 35, 36, 37, 38]

print(len(clothing_forms))
print(len(electronic_forms))
print(len(food_forms))

print("total participants:", len(clothing_forms) + len(electronic_forms) + len(food_forms))

28
33
26
total participants: 87
time: 7.77 ms (started: 2026-04-20 05:38:32 +00:00)


In [ ]:
from bs4 import BeautifulSoup, Comment

def extract_pairs(data):
    extracted_pairs = []

    for page in data.get('pages', []):
        pair_name = page.get('name')

        # Locate the HTML element
        html_element = next((el for el in page['elements'] if el['type'] == 'html'), None)

        if html_element:
            html_content = html_element['html']
            soup = BeautifulSoup(html_content, 'html.parser')

            # --- FIXED SECTION: Finding the marker without MARKER_RE.search ---
            is_randomised = "no"  # Default fallback

            # Search all HTML comments for the string "randomise"
            comments = soup.find_all(string=lambda text: isinstance(text, Comment))
            for comment in comments:
                if "randomise:" in comment:
                    # Clean up the string to get just 'yes' or 'no'
                    is_randomised = comment.replace("randomise:", "").strip().lower()
                    break
            # -----------------------------------------------------------------

            # Split by the two columns
            columns = soup.find_all('div', style=lambda x: x and 'flex:1' in x)

            pair_data = {
                "pair_id": pair_name,
                "is_randomised": is_randomised,
                "bundle_1": [],
                "bundle_2": []
            }

            for i, col in enumerate(columns):
                bundle_key = "bundle_1" if i == 0 else "bundle_2"
                items = col.find_all('div', style=lambda x: x and 'background:#fff' in x)

                for item in items:
                    name_div = item.find('div', style=lambda x: x and '700' in x)
                    desc_div = item.find('div', style=lambda x: x and '14px' in x)

                    if name_div and desc_div:
                        pair_data[bundle_key].append({
                            "item_name": name_div.get_text(strip=True),
                            "description": desc_div.get_text(strip=True)
                        })

            extracted_pairs.append(pair_data)

    return extracted_pairs

time: 2.93 ms (started: 2026-04-20 06:01:31 +00:00)


In [ ]:
def prepare_llm_input(extracted_pairs):
    bundle_strings = []
    randomisation_vector = []

    for pair in extracted_pairs:
        # 1. Build the string for Bundle 1 (Bundle A)
        b1_text = "Bundle A:\n"
        for item in pair['bundle_1']:
            b1_text += f"- [{item['item_name']}] - {item['description']}\n"

        # 2. Build the string for Bundle 2 (Bundle B)
        b2_text = "\nBundle B:\n"
        for item in pair['bundle_2']:
            b2_text += f"- [{item['item_name']}] - {item['description']}\n"

        # Combine them into one prompt block
        full_pair_string = f"{b1_text}{b2_text}" #Pair ID: {pair['pair_id']}

        bundle_strings.append(full_pair_string)
        randomisation_vector.append(pair['is_randomised'])

    return bundle_strings, randomisation_vector

# --- EXECUTION ---
# Assuming 'extracted_pairs' is the result from the previous extract_pairs function
prompts, random_map = prepare_llm_input(single_pair)

# Example output for the first pair
print("--- PROMPT VECTOR [0] ---")
print(prompts[0])
print(f"--- RANDOMISER VECTOR [0]: {random_map[0]} ---")

--- PROMPT VECTOR [0] ---
Bundle A:
- [Pink Queen Rhinestone Leopard Print Bikini Halter Top Hipster Large] - A stylish bikini set featuring a halter top with rhinestone detailing and a leopard print design, paired with hipster-style bottoms, all in a pink color scheme.
- [Women's Floral Halter Swimwear Bikini Set Padded Bra Bikini Swimsuit S~L] - This is a women's bikini set featuring a floral design, halter neck style, and padded bra, available in sizes small to large.

Bundle B:
- [Pink Queen Rhinestone Leopard Print Bikini Halter Top Hipster Large] - A stylish bikini set featuring a halter top with rhinestone detailing and a leopard print design, paired with hipster-style bottoms, all in a pink color scheme.
- [Women's Floral Halter Swimwear Bikini Set Padded Bra Bikini Swimsuit S~L] - This is a women's bikini set featuring a floral design, halter neck style, and padded bra, available in sizes small to large.
- [WIIPU sexy 3 row waist belly chain/Waist Chain/bikini body chain(wiipu

In [ ]:
def claude_bundle_prompt(bundles_here):
    llm_prompt = f"""Task: Compare original and modified item bundles (intended to be
purchased together). Bundle positions are randomised (A/B).

Part 1 - Choose the better designed bundle: Consider:
- Was the bundle designed with a clear idea in mind?
- Do the items make sense to purchase together?
Part 2 - Rate each bundle (1-5)

{bundles_here}
Score:
1-2: Low-quality: No/weak relations between bundle items.
3: Needs improvement: One/two modifications needed.
4-5: High-quality: Reasonable bundle to purchase.

**## OUTPUT FORMAT:**
Print your reasoning first. Then, print the separator string `===JSON_START===` on a new line. Finally, provide a single JSON object:

```json
{{
  "preferred_bundle": "string (single letter A/B)"
  "Bundle_A_rating: ": integer (1-5)
  "Bundle_B_rating": integer (1-5)
}}
```"""

    return llm_prompt

time: 773 µs (started: 2026-04-20 06:07:01 +00:00)


In [ ]:
print(claude_bundle_prompt(prompts[0]))


Task: Compare original and modified item bundles (intended to be
purchased together). Bundle positions are randomised (A/B).

Part 1 - Choose the better designed bundle: Consider:
- Was the bundle designed with a clear idea in mind?
- Do the items make sense to purchase together?
Part 2 - Rate each bundle (1-5)

Bundle A:
- [Pink Queen Rhinestone Leopard Print Bikini Halter Top Hipster Large] - A stylish bikini set featuring a halter top with rhinestone detailing and a leopard print design, paired with hipster-style bottoms, all in a pink color scheme.
- [Women's Floral Halter Swimwear Bikini Set Padded Bra Bikini Swimsuit S~L] - This is a women's bikini set featuring a floral design, halter neck style, and padded bra, available in sizes small to large.

Bundle B:
- [Pink Queen Rhinestone Leopard Print Bikini Halter Top Hipster Large] - A stylish bikini set featuring a halter top with rhinestone detailing and a leopard print design, paired with hipster-style bottoms, all in a pink col

In [ ]:

from google.colab import userdata


open_secret_key = userdata.get('open_router')

if open_secret_key:
  print("OpenRouter Token retrieved successfully.")
else:
  print("Token not found in Colab Secrets.")


from openai import OpenAI


import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=open_secret_key,
)


OpenRouter Token retrieved successfully.
time: 600 ms (started: 2026-04-20 06:38:30 +00:00)


In [ ]:
from tqdm.asyncio import tqdm_asyncio

async def openrouter_request(user, model_id, system=None):
    if system:
        message = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    else:
        message = [{"role": "user", "content": user}]

    # Robust retry loop with exponential backoff
    for delay_secs in (2**x for x in range(0, 3)):
        try:
            response = await async_client.chat.completions.create(
                model=model_id, # Now dynamic!
                messages=message,
                temperature=0,
                max_tokens=2000, # Adjust based on bundle length
                # Optional: extra_body is where OpenRouter specific features go
                extra_body={
                    "provider": {"require_parameters": True}
                }
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            randomness_collision_avoidance = random.randint(0, 1000) / 1000.0
            sleep_dur = delay_secs + randomness_collision_avoidance
            print(f"Error with {model_id}: {e}. Retrying in {round(sleep_dur, 2)}s.")
            await asyncio.sleep(sleep_dur)

    return None


async def run_experiment(prompts, model_id, system=None, batch_size=20):
    """
    Unified experiment runner with a real-time progress bar.
    It still respects the Semaphore 'bouncer' you set up globally.
    """
    results = []
    print(f"🚀 Initializing experiment for: {model_id}")

    # 1. Create all tasks immediately.
    # Your 'async with sem:' inside single_request will handle the
    # actual throttling so you don't hit rate limits.
    tasks = [
        openrouter_request(d["prompts"], model_id, system=system)
        for d in prompts
    ]

    # 2. Use tqdm_asyncio.gather to run all tasks with a live progress bar.
    # This replaces the manual 'for' loop and 'asyncio.gather' batches.
    results = await tqdm_asyncio.gather(
        *tasks,
        desc=f"📊 {model_id.split('/')[-1]}", # Shows model name (e.g., 'llama-3.3-70b')
        total=len(tasks)
    )

    print(f"✅ {model_id} — All {len(results)} requests completed.\n")
    return results

time: 5.41 ms (started: 2026-04-20 06:36:50 +00:00)


In [ ]:
import pickle as pkl

async def openrouter_run_tests(domain, json_numbers):

    system_message = "You are an **Expert E-commerce Bundle Analyst and Product Bundler**. Your task is to analyse two bundles and determine which one is better, with an associated score for each."

    baseline_models = [
        "anthropic/claude-3-5-haiku",

    ]

    names = ["claude"]

    batch_sizes = [40]

    prompt_list = []
    mapping = []

    for i in json_numbers:
        json_path = f"/content/drive/MyDrive/bundles/surveys/{domain}_batch_{str(i)}.json"         # f"/content/3_Human Evaluation/surveys/{domain}_batch_{str(i)}.json"
        with open(json_path, 'r') as f: # file path
            example_json = json.load(f)

        pairs = extract_pairs(example_json)

        prompts, random_map = prepare_llm_input(pairs)
        for j in range(20):
            prompt_list.append(claude_bundle_prompt(prompts[j]))
            mapping.append(random_map[j])



    claude_prompt_list = [{"prompts": p} for p in prompt_list]

    print("Number of prompts:", len(claude_prompt_list))


    # claude_prompt_list = [{"prompts": prompt_list[0]}]

    print(prompt_list[0])

    for i in range(0,1):
        zero_shot_responses = await run_experiment(claude_prompt_list, model_id=baseline_models[i], system=system_message, batch_size=batch_sizes[i])


        with open(f"/content/drive/MyDrive/bundles/{domain}_claude_zero_shot_responses.pkl", 'wb') as f:
            pkl.dump([zero_shot_responses, mapping], f)

    # print(prompt_list[0])
    print(zero_shot_responses[0])

time: 2.53 ms (started: 2026-04-20 06:43:10 +00:00)


In [ ]:
clothing_forms = [1, 3, 4, 5, 6, 7, 10, 11, 13, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 29, 30, 31, 32, 33, 34, 35, 37, 38]
electronic_forms = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 20, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 34, 35, 36, 39]
food_forms = [1, 2, 4, 5, 6, 7, 9, 10, 11, 12, 14, 15, 19, 20, 21, 23, 24, 27, 28, 31, 33, 34, 35, 36, 37, 38]


time: 1.43 ms (started: 2026-04-20 06:43:14 +00:00)


In [ ]:
await openrouter_run_tests("electronics", electronic_forms)

await openrouter_run_tests("clothing", clothing_forms)

await openrouter_run_tests("food", food_forms)

Number of prompts: 660

Task: Compare original and modified item bundles (intended to be
purchased together). Bundle positions are randomised (A/B).

Part 1 - Choose the better designed bundle: Consider:
- Was the bundle designed with a clear idea in mind?
- Do the items make sense to purchase together?
Part 2 - Rate each bundle (1-5)

Bundle A:
- [Wasabi Power Battery for Canon LP-E6 and Canon EOS 5D Mark II, EOS 5D Mark III, EOS 6D, EOS 7D, EOS 60D, EOS 60Da, EOS 70D] - A high-capacity rechargeable battery compatible with various Canon DSLR camera models, providing extended shooting time for photographers.
- [Belkin 3-Outlet Mini Travel Swivel Charger Surge Protector with Dual USB Ports, 5 Charging Outlets Total (1 AMP / 5 Watt)] - A compact surge protector offering three AC outlets and two USB ports for charging multiple devices simultaneously while traveling.
- [Laptop/Notebook Battery for Dell Inspiron] - A rechargeable power source specifically designed to replace or serve as a b

📊 claude-3-5-haiku: 100%|██████████| 660/660 [00:34<00:00, 18.98it/s]


✅ anthropic/claude-3-5-haiku — All 660 requests completed.

Reasoning:

Bundle A:
This bundle appears to be focused on power and charging solutions across multiple devices. It includes:
- A Canon camera battery
- A travel surge protector/charger
- A Dell laptop battery
- An iPhone battery case

While the items are all related to power and charging, the bundle lacks a cohesive theme. The devices are from different manufacturers and for different types of electronics, which makes the bundle feel scattered and less targeted.

Bundle B:
This bundle is specifically focused on Canon camera batteries. It includes:
- A Canon camera battery
- A 2-pack of the same Canon camera batteries with a charger

Bundle B is much more focused and purposeful. It provides a clear solution for photographers who need backup batteries and a charger for their Canon DSLR cameras. The items are directly related and make sense to purchase together.

===JSON_START===
{
  "preferred_bundle": "B",
  "Bundle_A_rating":

📊 claude-3-5-haiku: 100%|██████████| 560/560 [00:19<00:00, 29.38it/s]


✅ anthropic/claude-3-5-haiku — All 560 requests completed.

Reasoning:

Bundle A and Bundle B are nearly identical, with Bundle B having an additional item - a waist chain. Let's analyze the bundles:

Bundle A:
- Contains two bikini sets
- Both items are swimwear
- Potential issue: Duplicate/very similar items
- Lacks cohesive styling or complementary accessories

Bundle B:
- Contains the same two bikini sets as Bundle A
- Adds a waist chain accessory
- The waist chain complements the swimwear
- Provides an additional styling element
- Creates a more complete beach/poolside look

The addition of the waist chain in Bundle B elevates the bundle by:
1. Adding a decorative accessory
2. Providing styling versatility
3. Creating a more intentional, curated set

Part 1 - Bundle Design:
- Bundle A: Lacks clear design concept (duplicate items)
- Bundle B: Better designed with a complementary accessory

Part 2 - Ratings:
- Bundle A: 2/5 (duplicate items, no clear theme)
- Bundle B: 4/5 (cohesive

📊 claude-3-5-haiku: 100%|██████████| 520/520 [00:20<00:00, 25.49it/s]

✅ anthropic/claude-3-5-haiku — All 520 requests completed.

Reasoning:

Bundle A:
- Focused on Japanese rice and seaweed-related products
- All items are food-related and seem to complement each other
- Consistent theme of Asian, particularly Japanese, cuisine and snacks
- Appears to be designed for someone who enjoys Japanese flavors and rice-based meals

Bundle B:
- Mostly similar to Bundle A, but with two notable additions
- Faygo Rock & Rye Soda seems out of place and disrupts the Japanese food theme
- Moonstruck Chocolate adds another unrelated element
- The core Japanese food items remain, but the additional products feel random and disconnected

Part 1 - Bundle Design:
- Bundle A has a clear, cohesive design focused on Japanese rice and seaweed products
- Bundle B introduces unrelated items that break the thematic consistency

Part 2 - Ratings:
- Bundle A: 5/5 - Perfectly curated bundle with a clear culinary theme
- Bundle B: 2/5 - Disrupted by random soda and chocolate items th

#Responses have been provided

In [3]:
import pickle as pkl

# with open(f"/content/drive/MyDrive/bundles/electronics_claude_zero_shot_responses.pkl", 'rb') as f:
with open(f"/content/LLM4BEAR/3_Human Evaluation/electronics_claude_zero_shot_responses.pkl", 'rb') as f:
    electronic_responses, electronic_mapping = pkl.load(f)

# with open(f"/content/drive/MyDrive/bundles/clothing_claude_zero_shot_responses.pkl", 'rb') as f:
with open(f"/content/LLM4BEAR/3_Human Evaluation/clothing_claude_zero_shot_responses.pkl", 'rb') as f:
    clothing_responses, clothing_mapping = pkl.load(f)

# with open(f"/content/drive/MyDrive/bundles/food_claude_zero_shot_responses.pkl", 'rb') as f:
with open(f"/content/LLM4BEAR/3_Human Evaluation/food_claude_zero_shot_responses.pkl", 'rb') as f:
    food_responses, food_mapping = pkl.load(f)

time: 5.22 ms (started: 2026-04-24 02:31:33 +00:00)


In [4]:
print("Number of electronic_responses:", len(electronic_responses), "\nMapping:", len(electronic_mapping))
print(electronic_responses[0])
print()
print("Number of clothing_responses:", len(clothing_responses), "\nMapping:", len(clothing_mapping))
print(clothing_responses[0])
print()
print("Number of food_responses:", len(food_responses), "\nMapping:", len(food_mapping))
print(food_responses[0])

Number of electronic_responses: 660 
Mapping: 660
Reasoning:

Bundle A:
This bundle appears to be focused on power and charging solutions across multiple devices. It includes:
- A Canon camera battery
- A travel surge protector/charger
- A Dell laptop battery
- An iPhone battery case

While the items are all related to power and charging, the bundle lacks a cohesive theme. The devices are from different manufacturers and for different types of electronics, which makes the bundle feel scattered and less targeted.

Bundle B:
This bundle is specifically focused on Canon camera batteries. It includes:
- A Canon camera battery
- A 2-pack of the same Canon camera batteries with a charger

Bundle B is much more focused and purposeful. It provides a clear solution for photographers who need backup batteries and a charger for their Canon DSLR cameras. The items are directly related and make sense to purchase together.

===JSON_START===
{
  "preferred_bundle": "B",
  "Bundle_A_rating": 2,
  "Bun

In [5]:
def extract_json_simple_replace(response_text):
    if response_text is None:
        return None

    try:
        # 1. Use a case-insensitive split or check for the separator
        if "===JSON_START===" not in response_text:
            # Fallback: Try to find the first '{' anyway
            json_part = response_text
        else:
            json_part = response_text.split("===JSON_START===")[1]

        # 2. Find the boundaries
        first_brace = json_part.find('{')
        last_brace = json_part.rfind('}')

        if first_brace == -1 or last_brace == -1:
            return None

        # 3. Extract and clean
        json_string = json_part[first_brace : last_brace + 1].strip()

        # 4. Parse
        return json.loads(json_string)

    except Exception as e:
        print(f"Extraction error: {e}")
        return None

time: 1.08 ms (started: 2026-04-24 02:31:39 +00:00)


In [7]:
def process_extractions(responses, category_name):
    extracted_results = []

    for i, resp in enumerate(responses):
        try:
            # Try to extract the JSON
            extracted = extract_json_simple_replace(resp)
            extracted_results.append(extracted)
        except Exception as e:
            # If it fails, log the index and the error
            print(f"Error in {category_name} at index {i}: {e}")
            extracted_results.append(None)  # Placeholder

    return extracted_results

# Process all three
electronic_extraction = process_extractions(electronic_responses, "Electronics")
clothing_extraction = process_extractions(clothing_responses, "Clothing")
food_extraction = process_extractions(food_responses, "Food")

Extraction error: Expecting value: line 3 column 18 (char 20)
Extraction error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Extraction error: Invalid control character at: line 4 column 13 (char 18)
Extraction error: Expecting ':' delimiter: line 4 column 15 (char 20)
Extraction error: Expecting property name enclosed in double quotes: line 3 column 3 (char 5)
time: 10.4 ms (started: 2026-04-24 02:31:40 +00:00)


In [ ]:
for i in range(len(electronic_extraction)):
    print(i, electronic_extraction[i])

0 {'preferred_bundle': 'B', 'Bundle_A_rating': 2, 'Bundle_B_rating': 5}
1 {'preferred_bundle': 'A', 'Bundle_A_rating': 5, 'Bundle_B_rating': 2}
2 {'preferred_bundle': 'B', 'Bundle_A_rating': 3, 'Bundle_B_rating': 5}
3 {'preferred_bundle': 'A', 'Bundle_A_rating': 5, 'Bundle_B_rating': 1}
4 {'preferred_bundle': 'B', 'Bundle_A_rating': 3, 'Bundle_B_rating': 5}
5 {'preferred_bundle': 'B', 'Bundle_A_rating': 3, 'Bundle_B_rating': 5}
6 {'preferred_bundle': 'B', 'Bundle_A_rating': 2, 'Bundle_B_rating': 5}
7 {'preferred_bundle': 'B', 'Bundle_A_rating': 2, 'Bundle_B_rating': 5}
8 {'preferred_bundle': 'A', 'Bundle_A_rating': 5, 'Bundle_B_rating': 3}
9 {'preferred_bundle': 'B', 'Bundle_A_rating': 3, 'Bundle_B_rating': 4}
10 {'preferred_bundle': 'B', 'Bundle_A_rating': 2, 'Bundle_B_rating': 5}
11 {'preferred_bundle': 'A', 'Bundle_A_rating': 5, 'Bundle_B_rating': 2}
12 {'preferred_bundle': 'A', 'Bundle_A_rating': 5, 'Bundle_B_rating': 3}
13 {'preferred_bundle': 'A', 'Bundle_A_rating': 5, 'Bundle_B_

In [ ]:
for i in range(len(clothing_extraction)):
    print(i, clothing_extraction[i])

0 {'preferred_bundle': 'B', 'Bundle_A_rating': 2, 'Bundle_B_rating': 4}
1 {'preferred_bundle': 'B', 'Bundle_A_rating': 2, 'Bundle_B_rating': 5}
2 {'preferred_bundle': 'B', 'Bundle_A_rating': 3, 'Bundle_B_rating': 4}
3 {'preferred_bundle': 'A', 'Bundle_A_rating': 4, 'Bundle_B_rating': 2}
4 {'preferred_bundle': 'B', 'Bundle_A_rating': 2, 'Bundle_B_rating': 4}
5 {'preferred_bundle': 'A', 'Bundle_A_rating': 5, 'Bundle_B_rating': 2}
6 {'preferred_bundle': 'A', 'Bundle_A_rating': 5, 'Bundle_B_rating': 2}
7 {'preferred_bundle': 'A', 'Bundle_A_rating': 3, 'Bundle_B_rating': 2}
8 {'preferred_bundle': 'A', 'Bundle_A_rating': 2, 'Bundle_B_rating': 1}
9 {'preferred_bundle': 'A', 'Bundle_A_rating': 4, 'Bundle_B_rating': 3}
10 {'preferred_bundle': 'A', 'Bundle_A_rating': 4, 'Bundle_B_rating': 3}
11 {'preferred_bundle': 'A', 'Bundle_A_rating': 5, 'Bundle_B_rating': 2}
12 {'preferred_bundle': 'B', 'Bundle_A_rating': 2, 'Bundle_B_rating': 5}
13 {'preferred_bundle': 'A', 'Bundle_A_rating': 4, 'Bundle_B_

In [ ]:
for i in range(len(food_extraction)):
    print(i, food_extraction[i])

0 {'preferred_bundle': 'A', 'Bundle_A_rating': 5, 'Bundle_B_rating': 2}
1 {'preferred_bundle': 'B', 'Bundle_A_rating': 2, 'Bundle_B_rating': 5}
2 {'preferred_bundle': 'A', 'Bundle_A_rating': 5, 'Bundle_B_rating': 3}
3 {'preferred_bundle': 'A', 'Bundle_A_rating': 5, 'Bundle_B_rating': 4}
4 {'preferred_bundle': 'A', 'Bundle_A_rating': 4, 'Bundle_B_rating': 2}
5 {'preferred_bundle': 'B', 'Bundle_A_rating': 3, 'Bundle_B_rating': 4}
6 {'preferred_bundle': 'B', 'Bundle_A_rating': 3, 'Bundle_B_rating': 5}
7 {'preferred_bundle': 'A', 'Bundle_A_rating': 4, 'Bundle_B_rating': 3}
8 {'preferred_bundle': 'A', 'Bundle_A_rating': 4, 'Bundle_B_rating': 2}
9 {'preferred_bundle': 'B', 'Bundle_A_rating': 2, 'Bundle_B_rating': 5}
10 {'preferred_bundle': 'A', 'Bundle_A_rating': 5, 'Bundle_B_rating': 2}
11 {'preferred_bundle': 'A', 'Bundle_A_rating': 5, 'Bundle_B_rating': 3}
12 {'preferred_bundle': 'A', 'Bundle_A_rating': 5, 'Bundle_B_rating': 2}
13 {'preferred_bundle': 'B', 'Bundle_A_rating': 2, 'Bundle_B_

In [ ]:
electronic_errs = [440, 504]
clothing_errs = [81, 107, 253, 268]

time: 769 µs (started: 2026-04-20 06:55:34 +00:00)


In [ ]:
for i in electronic_errs:
    print(i, electronic_responses[i])

    print()
    print("---------------------------------------")
    print()



440 Reasoning:



Bundle A A:
- Contains Contains a of250GB Samsung SSD and a a 1/TB WBlueD
-/desktop storage solution with complementary storage technologies

-/desktop installation installation kit

- Clear purpose:2provide storage with fast SSD and large capacity H
DD
-/desktop storage with complementstorageary technologies

Bundle B:B:
:
- Contains same SAME items as Bundle A, but with an additional WD CavBlackiar 1 TB drive
-/Desktop storage solution

- Essentially identical to Bundle A with A with one extra hard drive

Preferred Bundle Analysis:
- -- provides a clean, focused focused storage solution
- has a balanced good mix of S/desktop storage technologies
-/Provides complementary storage options with clear purpose

Bundle B appears redundant with an extra hard drive that doesn't meaningfully add value improve the bundle

:
-/Preferred Bundle A
more streamlined and purposeful storage solution
bundle
technology pairing

===JSON_START===_START===
{

    "preferred_":_bundle": "A

In [ ]:
for i in clothing_errs:
    print(i, clothing_responses[i])

    print()
    print("---------------------------------------")
    print()



81 Reasoning:
Bundle A contains includes more items and appears to have a more diverse range of shoes, but lacks a cohesive theme theme theme. The The bundle mixes children's shoes with a women's shoe,, (Calvin Klein pump pump), which includes seems disjoizedinted.

Bundle B B Bundle is more more focused, containing three children's shoes that appear to be part from the same brand/style family (LilIz Bundle seems more intentionally curated, with more alear target demographic (children shoes).Part The items look like they could be purchased could be part of a coordcoordinated shoe collection for for children.

The key differences::
- Bundle A has 5 items,, with mixed demographics

B has   3 with more consistent children's shoe theme selection


Part  1 Assessment:
- Bundle B B appears better designed, with on alear theme and more.

Part   Rating:
- Bundle A:: A 3/5 ((mixed items, lacks lacks clear focus)

- Bundle B: B:: 4 /5 (( more coherent, child-focused))===JSON_START===_START===
{


In [13]:
electronic_extraction[440] = {'preferred_bundle': 'A', 'Bundle_A_rating': 4, 'Bundle_B_rating': 2} # asked again bundle b = 2
electronic_extraction[504] = {'preferred_bundle': 'A', 'Bundle_A_rating': 5, 'Bundle_B_rating': 3}

clothing_extraction[81] = {'preferred_bundle': 'B', 'Bundle_A_rating': 3, 'Bundle_B_rating': 4}
clothing_extraction[107] = {'preferred_bundle': 'A', 'Bundle_A_rating': 5, 'Bundle_B_rating': 2}
clothing_extraction[253] = {'preferred_bundle': 'B', 'Bundle_A_rating': 3, 'Bundle_B_rating': 5}
clothing_extraction[268] = {'preferred_bundle': 'B', 'Bundle_A_rating': 2, 'Bundle_B_rating': 4}


time: 1.12 ms (started: 2026-04-24 02:32:39 +00:00)


dict

time: 8.96 ms (started: 2026-04-20 07:01:37 +00:00)


{'preferred_bundle': 'A', 'Bundle_A_rating': 4, 'Bundle_B_rating': 2}

time: 9.15 ms (started: 2026-04-20 07:05:17 +00:00)


In [ ]:
electronic_mapping[440]

'yes'

time: 2.18 ms (started: 2026-04-20 07:05:37 +00:00)


In [14]:
def get_final_scores(extractions, mapping):
    original_scores = []
    modified_scores = []
    preferences = [] # To track if the LLM liked the 'Original' or 'Modified' more

    for i in range(len(extractions)):
        # Skip if extraction failed (None)
        if extractions[i] is None:
            original_scores.append(None)
            modified_scores.append(None)
            preferences.append(None)
            continue

        data = extractions[i]
        swapped = (mapping[i] == 'yes')

        rating_a = data['Bundle_A_rating']
        rating_b = data['Bundle_B_rating']
        pref = data['preferred_bundle'] # 'A' or 'B'

        if not swapped:
            # A is Original, B is Modified
            orig = rating_a
            mod = rating_b
            winner = "Original" if pref == 'A' else "Modified"
        else:
            # A is Modified, B is Original
            orig = rating_b
            mod = rating_a
            winner = "Original" if pref == 'B' else "Modified"

        original_scores.append(orig)
        modified_scores.append(mod)
        preferences.append(winner)

    return original_scores, modified_scores, preferences

# --- EXECUTION ---
elec_orig, elec_mod, elec_prefs = get_final_scores(electronic_extraction, electronic_mapping)
cloth_orig, cloth_mod, cloth_prefs = get_final_scores(clothing_extraction, clothing_mapping)
food_orig, food_mod, food_prefs = get_final_scores(food_extraction, food_mapping)

time: 2.61 ms (started: 2026-04-24 02:32:42 +00:00)


In [15]:
print("Avg Original Electronic Score:", np.mean(elec_orig), "\nAvg Modified Electronic Score:", np.mean(elec_mod))
print("Avg Original Clothing Score:", np.mean(cloth_orig), "\nAvg Modified Clothing Score:", np.mean(cloth_mod))
print("Avg Original Food Score:", np.mean(food_orig), "\nAvg Modified Food Score:", np.mean(food_mod))

Avg Original Electronic Score: 2.7818181818181817 
Avg Modified Electronic Score: 4.534848484848485
Avg Original Clothing Score: 3.0642857142857145 
Avg Modified Clothing Score: 3.742857142857143
Avg Original Food Score: 3.1826923076923075 
Avg Modified Food Score: 4.282692307692308
time: 4.01 ms (started: 2026-04-24 02:32:43 +00:00)


In [10]:
len(elec_orig)

660

time: 3.52 ms (started: 2026-04-24 02:32:14 +00:00)


In [ ]:
def screen(vec):

    bads = 0
    for i in vec:
        # print(i)
        if i < 4:
            bads += 1

    total_l = len(vec)
    print(bads)

    return (total_l - bads) / total_l * 100

time: 588 µs (started: 2026-04-23 06:16:42 +00:00)


In [ ]:
print("Bundlerec Elec:",screen(elec_orig))
print("Bundlerec Cloth:",screen(cloth_orig))
print("Bundlerec Food:",screen(food_orig))

print()

print("LLM4BEAR Elec:",screen(elec_mod))
print("LLM4BEAR Cloth:",screen(cloth_mod))
print("LLM4BEAR Food:",screen(food_mod))

522
Bundlerec Elec: 20.909090909090907
360
Bundlerec Cloth: 35.714285714285715
341
Bundlerec Food: 34.42307692307692

71
LLM4BEAR Elec: 89.24242424242425
202
LLM4BEAR Cloth: 63.92857142857142
111
LLM4BEAR Food: 78.65384615384615
time: 18.7 ms (started: 2026-04-23 06:23:05 +00:00)
